# Classification Model Decision Trees, Bagging, Boosting & Model Tuning

**Business case:** A bank wants to identify customers who may leave (churn). We will build classification models, compare training vs. test performance, inspect ROC–AUC, interpret a decision tree, and tune hyperparameters without leaking information from the test set.

**Models covered**
1. Decision Tree
2. Bagging Classifier (many trees trained on bootstrap samples)
3. Random Forest (a practical bagging-based ensemble)
4. AdaBoost (boosting)
5. Gradient Boosting (boosting)

**Dataset:** IBM Telco Customer Churn, loaded from a public CSV. The target is `Churn` (Yes/No). This is a realistic customer-retention dataset with customer profile, service, contract, billing, and tenure features.

> Internet access is required for the first dataset download. If the URL is unavailable, download the CSV from the IBM Telco Customer Churn sample dataset and update `DATA_URL` below.


## Notebook navigation

1. **Understand the data:** load, audit, split and preprocess.
2. **Train classifiers:** Decision Tree, Bagging, Random Forest, AdaBoost and Gradient Boosting.
3. **Evaluate models:** training/testing metrics, ROC–AUC, confusion matrix and feature importance.
4. **Estimate reliability:** stratified cross-validation.
5. **Tune a model:** GridSearchCV, best parameters and final test evaluation.
6. **Discuss decisions:** thresholds, overfitting and project-quality checks.

**How to study each section:** read the markdown first, run the code cell, inspect the output, and explain the result in your own words. The additional notes explain what the important code lines and parameters do.

## 1. Learning outcomes
By the end, you should be able to:
- Prepare a classification dataset and avoid target leakage.
- Train a baseline tree and explain its splits.
- Distinguish bagging from boosting.
- Evaluate both training and testing performance.
- Compare confusion matrix, precision, recall, F1, ROC–AUC, and ROC curves.
- Use cross-validation and `GridSearchCV` to tune a model responsibly.
- Explain why a model with near-perfect training scores may generalize poorly.


## 2. Setup and imports
We use a `Pipeline` for preprocessing so the same transformations are fitted correctly inside each cross-validation fold. This is a good habit even in a teaching notebook.

In [ ]:
# If needed, uncomment this line in Jupyter / Colab:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 3. Load the real-world dataset
The CSV has one row per customer. `Churn` is the outcome we want to predict.

**Target meaning**
- `Churn = Yes`: customer left.
- `Churn = No`: customer stayed.

We drop `customerID` because it is an identifier, not a meaningful predictive feature. We also remove `TotalCharges` blank strings and let the imputer handle missing values.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/ telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv".replace(" ", "")

# Alternative commonly mirrored public sample:
# DATA_URL = "https://raw.githubusercontent.com/IBM/Customer-Churn-Data/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
display(df.head())


### Quick data audit
Before modeling, inspect types, missingness, target balance, and duplicate rows. These checks often reveal issues that change the modeling plan.

In [ ]:
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Duplicate rows:", df.duplicated().sum())

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

display(df["Churn"].value_counts(dropna=False).to_frame("count"))
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

sns.countplot(data=df, x="Churn")
plt.title("Target distribution: customer churn")
plt.show()


## 4. Define features and target
We map `Yes` to 1 and `No` to 0. Positive class (1) means **churn**. This matters because precision, recall, and the ROC curve will focus on identifying customers who churn.

We split before fitting imputers/encoders. The test set remains untouched until final evaluation.

In [ ]:
df = df.drop_duplicates().copy()
df = df.dropna(subset=["Churn"])

X = df.drop(columns=["Churn", "customerID"], errors="ignore")
y = df["Churn"].map({"No": 0, "Yes": 1})

# Remove any rows with an unmapped target, if present
valid = y.notna()
X, y = X.loc[valid].copy(), y.loc[valid].astype(int)

# stratify=y keeps the target proportions similar in train and test.
# test_size=0.20 reserves 20% for final evaluation; random_state makes the split repeatable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("\nChurn rate — train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3))


## 5. Preprocessing with a ColumnTransformer
Most columns are categorical; `tenure`, `MonthlyCharges`, and `TotalCharges` are numeric.

- **Numeric:** median imputation.
- **Categorical:** most-frequent imputation, then one-hot encoding.
- `handle_unknown="ignore"` allows a category appearing only in test data without crashing.
- Preprocessing lives inside the pipeline, so cross-validation learns preprocessing from each training fold only.

Decision trees do not require feature scaling, so we do not standardize numeric columns here.

### How preprocessing works

This cell prepares the data before modeling.

- `select_dtypes(...)` automatically separates numeric columns from categorical columns.
- `SimpleImputer(strategy="median")` fills missing numeric values using the median.
- `SimpleImputer(strategy="most_frequent")` fills missing categorical values using the most common category.
- `OneHotEncoder(handle_unknown="ignore")` converts categories into numeric indicator columns. `ignore` prevents errors if a category appears in test data but not training data.
- `ColumnTransformer` applies the numeric and categorical steps to the appropriate columns.
- `Pipeline` keeps the steps together so preprocessing is fitted correctly inside cross-validation.

**Why this matters:** preprocessing should learn from training data only. The pipeline helps prevent data leakage.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical feature count:", len(categorical_features))


## 6. Build the models
### Decision Tree
Learns a sequence of feature-based questions. `max_depth` limits tree depth; `min_samples_leaf` requires a minimum number of rows in each leaf; `criterion` measures split quality.

### Bagging
Fits multiple base estimators on bootstrap samples and aggregates their predictions. It primarily reduces variance. We explicitly use decision trees as base estimators.

### Random Forest
A bagging-based tree ensemble that also considers a random subset of features at each split. This reduces correlation between trees.

### AdaBoost
Sequentially fits weak learners, giving more attention to previously misclassified observations. Here we use shallow decision stumps/trees.

### Gradient Boosting
Adds trees sequentially to improve the current ensemble's errors, using a loss function and a learning rate. Smaller learning rates often need more estimators.

### Model parameters explained

**Decision Tree**
- `criterion="gini"`: measures how mixed the classes are at a split; `entropy` is an alternative.
- `max_depth=5`: limits tree depth to help control overfitting.
- `min_samples_leaf=10`: each leaf must contain at least 10 training rows.
- `class_weight="balanced"`: gives extra weight to the minority class.
- `random_state=42`: makes randomized behavior reproducible.

**Bagging**
- `estimator=DecisionTreeClassifier(...)`: specifies the model used for each ensemble member.
- `n_estimators=100`: builds 100 trees.
- `max_samples=0.8`: samples about 80% of rows for each tree.
- `bootstrap=True`: samples rows with replacement.
- `n_jobs=-1`: uses available CPU cores.

**Random Forest**
- `n_estimators=200`: number of trees.
- `max_features="sqrt"`: considers a random subset of features at each split.
- `min_samples_leaf=2`: prevents leaves from being based on a single row.
- `class_weight="balanced"`: accounts for class imbalance during fitting.

**AdaBoost**
- `estimator=DecisionTreeClassifier(max_depth=1)`: uses simple one-split trees as weak learners.
- `learning_rate=0.5`: controls how much each learner contributes.
- `n_estimators=100`: maximum number of boosting stages.

**Gradient Boosting**
- `n_estimators=100`: number of sequential stages.
- `learning_rate=0.05`: contribution of each new tree.
- `max_depth=2`: keeps component trees shallow.
- `min_samples_leaf=10`: requires enough rows in each leaf.

These are teaching-friendly starting values, not guaranteed best settings.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, criterion="gini",
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=None, min_samples_leaf=2, random_state=RANDOM_STATE
        ),
        n_estimators=100, max_samples=0.8, max_features=1.0,
        bootstrap=True, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=2,
        min_samples_leaf=10, random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    for name, model in models.items()
}


## 7. Train all models and compare train vs. test
We report:
- **Accuracy:** overall fraction correct.
- **Precision:** of customers predicted to churn, how many actually churned?
- **Recall:** of all customers who churned, how many did we identify?
- **F1:** harmonic mean of precision and recall.
- **ROC–AUC:** ranking ability across thresholds; 0.5 is random ranking and 1.0 is perfect ranking.

Training scores diagnose fit; test scores estimate generalization. A large train–test gap can indicate overfitting. For churn, recall may be especially important if missing a likely-to-leave customer is costly, but business costs should determine the final metric.

### How model training and scoring work

`pipe.fit(X_train, y_train)` learns preprocessing settings and model patterns from training data.

Inside `get_scores()`:
1. `predict()` produces class labels: 0 = stayed, 1 = churned.
2. `predict_proba()[:, 1]` returns predicted probabilities for churn (class 1).
3. The metric functions compare predictions with the actual labels.

**Metrics**
- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** of customers predicted to churn, how many actually churned?
- **Recall:** of all churners, how many did the model find?
- **F1:** balances precision and recall.
- **ROC–AUC:** measures how well the model ranks churners above non-churners across thresholds.

Training scores show fit to seen data. Test scores estimate performance on unseen data. A large gap can indicate overfitting.

In [ ]:
def get_scores(pipe, X_part, y_part):
    pred = pipe.predict(X_part)
    proba = pipe.predict_proba(X_part)[:, 1]
    return {
        "Accuracy": accuracy_score(y_part, pred),
        "Precision": precision_score(y_part, pred, zero_division=0),
        "Recall": recall_score(y_part, pred, zero_division=0),
        "F1": f1_score(y_part, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_part, proba)
    }

fitted_models = {}
rows = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    train_scores = get_scores(pipe, X_train, y_train)
    test_scores = get_scores(pipe, X_test, y_test)

    row = {"Model": name}
    for metric, value in train_scores.items():
        row[f"Train {metric}"] = value
    for metric, value in test_scores.items():
        row[f"Test {metric}"] = value
    rows.append(row)

results = pd.DataFrame(rows).set_index("Model").sort_values("Test ROC-AUC", ascending=False)
display(results.round(3))


### Visual comparison: test metrics
Use this to discuss trade-offs rather than choosing a model based on accuracy alone.

### Reading the model comparison chart

Each bar shows a test-set metric. Higher is better for these metrics, but the best model depends on the business goal. For churn, recall may matter because missed churners are costly; precision also matters because unnecessary retention offers cost money. Compare several metrics rather than choosing by accuracy alone.

In [ ]:
test_cols = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
results[test_cols].plot(kind="bar", figsize=(14, 6), ylim=(0, 1.05))
plt.title("Test-set metric comparison")
plt.ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. ROC curves — training and testing, side by side
The ROC curve plots true positive rate (recall) against false positive rate at different probability thresholds. AUC summarizes ranking performance across thresholds.

**Teaching point:** training AUC can look excellent while test AUC is lower. The gap can be a sign of overfitting. Do not use test performance repeatedly to tune the model; use cross-validation on training data for model selection.

### ROC curves: training versus testing

`predict_proba(... )[:, 1]` supplies churn probabilities. `roc_curve()` calculates true-positive rate (recall) and false-positive rate at different thresholds.

- **TPR:** fraction of actual churners correctly found.
- **FPR:** fraction of customers who stayed but were incorrectly flagged as churners.
- **AUC:** summarizes ranking quality; 0.5 is approximately random and 1.0 is perfect.

The left panel is training performance; the right panel is test performance. A much better training curve than test curve may indicate overfitting. Keep the test set for final evaluation rather than repeatedly tuning against it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for name, pipe in fitted_models.items():
    train_prob = pipe.predict_proba(X_train)[:, 1]
    fpr, tpr, _ = roc_curve(y_train, train_prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_train, train_prob):.3f})")

    test_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, test_prob)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, test_prob):.3f})")

for ax, title in zip(axes, ["Training ROC curves", "Testing ROC curves"]):
    ax.plot([0, 1], [0, 1], "k--", label="Random baseline")
    ax.set_title(title)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Confusion matrix and classification report
A confusion matrix shows true negatives, false positives, false negatives, and true positives. For churn, a false negative is a customer who churns but was predicted to stay.

The default classification threshold is 0.5. This is not always the best business threshold; it should be selected using validation data and the relative costs of false positives and false negatives.

### Confusion matrix and classification report

The report shows precision, recall, F1 and support for each class. In the confusion matrix:
- **True negative:** correctly predicted a customer would stay.
- **False positive:** predicted churn, but the customer stayed.
- **False negative:** missed a customer who churned.
- **True positive:** correctly identified a churner.

This cell chooses a model by test AUC for demonstration. In a rigorous workflow, select models using cross-validation on training data and use the test set only for the final check.

In [ ]:
best_initial_name = results["Test ROC-AUC"].idxmax()
best_initial_model = fitted_models[best_initial_name]
test_pred = best_initial_model.predict(X_test)

print("Best initial model by test ROC-AUC:", best_initial_name)
print(classification_report(y_test, test_pred, target_names=["Stayed (0)", "Churned (1)"], zero_division=0))

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted stayed", "Predicted churn"],
            yticklabels=["Actual stayed", "Actual churn"])
plt.title(f"Test confusion matrix — {best_initial_name}")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.show()


## 10. Visualize the Decision Tree
We visualize the standalone decision tree, not an ensemble. Each node reports the split, Gini/entropy impurity, sample count, and class distribution. Colors show the majority predicted class.

The displayed tree is limited to depth 3 for readability; the trained model can be deeper. Feature names come from the fitted preprocessing pipeline.

### Reading the decision tree plot

`named_steps["preprocess"]` retrieves the fitted preprocessing stage; `named_steps["model"]` retrieves the trained tree. `get_feature_names_out()` returns names after one-hot encoding.

In `plot_tree()`:
- `feature_names` labels the split features.
- `class_names` labels the target classes.
- `filled=True` colors nodes by predicted class.
- `max_depth=3` displays only the first three levels for readability; it does not retrain or truncate the actual model.

Each node shows the split rule, impurity, number of samples, and class counts.

In [ ]:
tree_pipe = fitted_models["Decision Tree"]
fitted_preprocessor = tree_pipe.named_steps["preprocess"]
tree_model = tree_pipe.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

plt.figure(figsize=(24, 12))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=["Stayed", "Churned"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title("Decision Tree (first 3 levels shown)")
plt.show()

print("Text rules for the first levels:")
print(export_text(tree_model, feature_names=list(feature_names), max_depth=3))


## 11. Feature importance — useful, but interpret carefully
Tree-based importance measures how much a feature reduces impurity across splits. One-hot encoding means a categorical feature may be represented by multiple columns. Importance can be biased toward features with many possible splits and is not causal evidence.

For stronger interpretation, consider permutation importance on a held-out set, partial dependence, and domain validation.

### Feature importance: what it means

`feature_importances_` measures how much transformed features reduce impurity across the Random Forest. A larger score means a feature contributed more to splits in this fitted model.

Categorical columns may appear as multiple one-hot encoded features. Importance is useful for exploration, but it is not evidence of causation and can favor features with many possible splits.

In [ ]:
rf_pipe = fitted_models["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
rf_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()

importance = pd.Series(rf_model.feature_importances_, index=rf_names).sort_values(ascending=False).head(15)
display(importance.to_frame("Impurity-based importance"))

importance.sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Random Forest — top 15 transformed features")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 12. Cross-validation: what it is and when to use it
A single train/test split can be lucky or unlucky. **K-fold cross-validation** divides the training data into K folds:
1. Train on K−1 folds.
2. Validate on the remaining fold.
3. Repeat until each fold has served as validation.
4. Average the validation scores.

For classification, `StratifiedKFold` keeps the class proportion approximately similar in each fold. We use 5 folds and ROC–AUC, but the scoring metric should reflect the business goal.

**Important:** cross-validation is performed on `X_train` only. The test set remains reserved for the final check.

### Cross-validation settings

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` creates five folds while keeping the churn proportion approximately similar in each fold.

`cross_validate()`:
- `cv=cv`: uses the five-fold strategy.
- `scoring={...}`: calculates AUC, F1 and recall for each validation fold.
- `return_train_score=True`: saves training scores to help diagnose overfitting.
- `n_jobs=-1`: parallelizes work where possible.

The pipeline fits imputation and encoding separately inside each fold. Only training data is used; the held-out test set is not involved.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring={"auc": "roc_auc", "f1": "f1", "recall": "recall"},
        n_jobs=-1,
        return_train_score=True
    )
    cv_rows.append({
        "Model": name,
        "CV Train AUC Mean": scores["train_auc"].mean(),
        "CV Validation AUC Mean": scores["test_auc"].mean(),
        "CV Validation AUC Std": scores["test_auc"].std(),
        "CV Validation F1 Mean": scores["test_f1"].mean(),
        "CV Validation Recall Mean": scores["test_recall"].mean()
    })

cv_results = pd.DataFrame(cv_rows).set_index("Model").sort_values("CV Validation AUC Mean", ascending=False)
display(cv_results.round(3))


### How to interpret the CV table
- **Mean validation AUC:** average performance across folds.
- **Standard deviation:** stability across folds; smaller can indicate more consistent performance.
- **Train AUC much higher than validation AUC:** possible overfitting.
- **CV result vs. test result:** they should be broadly consistent, though they will not be identical.

Cross-validation estimates performance more robustly than one validation split, but it does not guarantee future production performance.

### Reading cross-validation results

The chart compares average validation scores across folds. A smaller standard deviation means scores were more consistent across folds; a large standard deviation can mean performance depends heavily on the split.

Cross-validation is a more stable estimate than a single split, but it is still an estimate of future performance.

In [ ]:
cv_results[["CV Validation AUC Mean", "CV Validation AUC Std", "CV Validation F1 Mean", "CV Validation Recall Mean"]].plot(
    kind="bar", figsize=(13, 5)
)
plt.title("Cross-validation summary (training data only)")
plt.ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 13. GridSearchCV: systematic hyperparameter tuning
`GridSearchCV` evaluates every parameter combination supplied in `param_grid` using cross-validation and chooses the combination with the best mean validation score.

**Core parameters**
- `estimator`: pipeline/model to tune.
- `param_grid`: dictionary of parameter names and candidate values. Pipeline parameters use `step__parameter`.
- `scoring`: metric to optimize, here ROC–AUC.
- `cv`: cross-validation strategy.
- `n_jobs`: parallel jobs; `-1` uses available CPU cores.
- `refit=True`: retrains the best configuration on all training data after search.
- `verbose`: progress messages.
- `return_train_score`: retains training scores to help diagnose overfitting.

Grid search can be computationally expensive. Start with a small, purposeful grid; expand only if needed.

### GridSearchCV: parameters and how the search works

Grid search tries every combination in `param_grid` using cross-validation, then selects the combination with the best mean validation ROC–AUC.

**Tree parameters in the grid**
- `criterion`: split-quality measure (`gini` or `entropy`).
- `max_depth`: maximum tree depth; `None` allows the tree to grow until other stopping rules apply.
- `min_samples_split`: minimum rows required before a node can split.
- `min_samples_leaf`: minimum rows allowed in a leaf.
- `max_features`: number/type of features considered at each split.

Because the model is in a pipeline, parameter names use `model__`, for example `model__max_depth`.

**GridSearchCV settings**
- `estimator`: complete preprocessing-plus-model pipeline.
- `param_grid`: values to test.
- `scoring="roc_auc"`: metric used to rank configurations.
- `cv=cv`: five-fold stratified validation.
- `refit=True`: refits the best configuration on all training data.
- `return_train_score=True`: saves training scores.
- `verbose=1`: prints progress.

This grid has 2 × 4 × 3 × 3 × 2 = **144 combinations**. Five-fold CV means 720 fits plus a final refit. Reduce the candidate lists for a faster classroom run.

In [ ]:
# Tune a Decision Tree first: the parameter effects are easy to visualize and explain.
tree_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))
])

param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_split": [2, 10, 20],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": [None, "sqrt"]
}

grid_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=True
)

# Grid search sees training data only; the held-out test set is not used to tune parameters.
grid_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))
print("Best parameters:")
display(pd.Series(grid_search.best_params_, name="Value").to_frame())


### GridSearchCV result inspection
The `cv_results_` attribute contains the mean and standard deviation of validation scores for each combination, as well as fit times and (when requested) training scores.

### Inspecting the grid-search results

`cv_results_` stores details for every combination:
- `mean_train_score`: average training-fold score.
- `mean_test_score`: average validation-fold score; this determines the winner.
- `std_test_score`: variation across validation folds.
- `mean_fit_time`: average fitting time.
- `params`: the tested parameter values.

A high training score with a much lower validation score may indicate overfitting. Prefer a strong and stable validation result, not just a complex tree.

In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)
display(
    grid_results[
        ["mean_train_score", "mean_test_score", "std_test_score", "mean_fit_time", "params"]
    ].sort_values("mean_test_score", ascending=False).head(10).round(4)
)

print("Number of parameter combinations:", len(grid_results))


## 14. Evaluate the tuned tree on the untouched test set
This is the first time we use test performance to evaluate the tuned model. We do not select hyperparameters based on test results.

Compare the tuned model against the initial Decision Tree baseline.

### Comparing the baseline and tuned tree

`best_estimator_` is the full pipeline refitted with the best parameters found during cross-validation. The table compares baseline and tuned trees on training and test data.

The search chooses settings using cross-validation; test performance is the final independent check. Tuning does not guarantee every metric will improve.

In [ ]:
tuned_tree = grid_search.best_estimator_

baseline_tree = fitted_models["Decision Tree"]
comparison_rows = []

for name, pipe in [
    ("Baseline Decision Tree", baseline_tree),
    ("Tuned Decision Tree", tuned_tree)
]:
    for split_name, X_part, y_part in [
        ("Train", X_train, y_train),
        ("Test", X_test, y_test)
    ]:
        scores = get_scores(pipe, X_part, y_part)
        comparison_rows.append({"Model": name, "Split": split_name, **scores})

tuning_comparison = pd.DataFrame(comparison_rows)
display(tuning_comparison.round(3))


## 15. Plot tuned tree ROC curves and the final tree
The train curve shows fit to data used for training. The test curve estimates performance on held-out data. A very high training AUC with weaker test AUC is a warning to consider stronger regularization or more data.

### Interpreting the tuned tree plots

The ROC plots compare ranking performance on training and test data. The tree diagram displays only the first three levels to keep it readable. Check `grid_search.best_params_` to connect the chosen depth and other settings to the tree structure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for split_name, X_part, y_part, ax in [
    ("Training", X_train, y_train, axes[0]),
    ("Testing", X_test, y_test, axes[1])
]:
    prob = tuned_tree.predict_proba(X_part)[:, 1]
    fpr, tpr, _ = roc_curve(y_part, prob)
    auc = roc_auc_score(y_part, prob)
    ax.plot(fpr, tpr, label=f"Tuned tree (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", label="Random baseline")
    ax.set_title(f"{split_name} ROC curve")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

tuned_model = tuned_tree.named_steps["model"]
tuned_names = tuned_tree.named_steps["preprocess"].get_feature_names_out()
plt.figure(figsize=(24, 12))
plot_tree(
    tuned_model,
    feature_names=tuned_names,
    class_names=["Stayed", "Churned"],
    filled=True, rounded=True, max_depth=3, fontsize=8
)
plt.title("Tuned Decision Tree (first 3 levels shown)")
plt.show()


## 16. Compare baseline and tuned tree — train/test score table
A tuning process should improve cross-validated performance, but it may not improve every metric or every split. The CV score is what selected the model; the test score is the final independent check.

### Compare train and test results

Look for changes in test ROC–AUC, precision, recall and F1. Also compare the training-to-test gap. A tuned model is not automatically better on every metric; in this search, ROC–AUC is the metric used to select parameters.

In [ ]:
display(
    tuning_comparison.pivot(index="Model", columns="Split", values=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]).round(3)
)


## 17. Optional: adjust the classification threshold
`predict()` normally uses a 0.5 threshold. If the business wants to catch more potential churners, a lower threshold may increase recall while also increasing false positives.

Do not choose the threshold using the final test set in a production workflow. Use a validation set or out-of-fold predictions and then report final test results once.

### Classification threshold: an important distinction

A threshold converts a predicted probability into a class label. For example, a threshold of `0.30` labels a customer as likely to churn when the predicted probability is at least 30%.

A lower threshold often raises recall but may reduce precision. ROC–AUC does not change with the threshold because it evaluates ranking across thresholds.

This is an illustration on test data. For a production workflow, choose a threshold using validation or out-of-fold predictions, then evaluate once on the untouched test set.

In [ ]:
# Illustration only: compare two thresholds on the test set.
# For a production workflow, choose the threshold on validation / out-of-fold predictions instead.
test_prob = tuned_tree.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in [0.30, 0.40, 0.50, 0.60]:
    pred = (test_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, test_prob)
    })

display(pd.DataFrame(threshold_rows).round(3))


## 18. Senior-level modeling checklist
Before trusting a classification model, ask:

1. **Is the target definition correct?** Confirm what “churn” means and when it is measured.
2. **Is there leakage?** Exclude identifiers and any fields only known after the outcome.
3. **Is the split realistic?** If future deployment predicts future churn, a time-based split may be more realistic than a random split.
4. **Is the metric aligned with the business?** ROC–AUC measures ranking, not the business cost of mistakes.
5. **Are classes imbalanced?** Review class proportions, precision, recall, PR–AUC, and confusion matrices.
6. **Are results stable?** Use cross-validation and examine standard deviation.
7. **Is the model interpretable enough?** A small tree is explainable; ensembles usually trade transparency for predictive power.
8. **Are hyperparameters tuned without touching test data?** Use CV on training data; evaluate the test set at the end.
9. **Can predictions be acted upon?** A churn score is useful only if the retention team has an appropriate intervention.
10. **Is the result reproducible?** Set random seeds, record package versions, and document data provenance.

### Key takeaway
- **Decision Tree:** easy to explain; can overfit.
- **Bagging:** parallel models on bootstrap samples; reduces variance.
- **Random Forest:** bagging plus feature randomness; strong general-purpose baseline.
- **Boosting:** sequentially improves weak learners; can capture complex patterns but needs tuning.
- **Cross-validation:** estimates how reliably a model generalizes.
- **GridSearchCV:** selects hyperparameters using CV, not the test set.


## 19. practice tasks
1. Change `max_depth` of the baseline tree to 2, 5, 10, and `None`. Compare train/test AUC.
2. Change `n_estimators` in Bagging from 25 to 100 to 200. Does test AUC improve consistently?
3. Compare AdaBoost `learning_rate` values 0.1, 0.5, and 1.0.
4. Explain why Random Forest may generalize better than one unrestricted tree.
5. Choose whether recall or precision is more important for the retention team and justify your choice.
6. Add `average_precision` as a cross-validation scoring metric and compare it with ROC–AUC.
7. Create a smaller grid and tune the Random Forest. Remember that the pipeline parameter names start with `model__` or the relevant step name.
